# Lesson 05 — Probing and Analysis: Did It Work?

**Read first:** `docs/06_evaluation_and_probing.md`

The paper's central claim is *versatility*: one frozen backbone, good at **appearance** (Kinetics) AND **motion** (Something-Something-v2). Our generator gives us ground truth for both factors, so we reproduce the claim in miniature:

| Probe | Analogue | Chance |
|---|---|---|
| digit class (10-way) | K400 / appearance | 10% |
| motion direction (8-way) | SSv2 / motion | 12.5% |

against the baseline that keeps everyone honest: a **randomly initialized** encoder (random ViT features are famously non-trivial).

Needs the checkpoint from lesson 04 (`checkpoints/vjepa_final.pt`, or upload/mount yours).

In [ ]:
#@title Setup — run this first { display-mode: "form" }
# Works both on Colab (clones the repo) and locally (repo already present).
import os, sys

if os.path.exists("../src/vjepa_mini"):          # running locally from notebooks/
    sys.path.insert(0, os.path.abspath("../src"))
elif os.path.exists("world-model-jepa/src"):      # already cloned on Colab
    sys.path.insert(0, os.path.abspath("world-model-jepa/src"))
else:                                             # fresh Colab runtime
    REPO_URL = "https://github.com/josephlionel95-moon/world-model-jepa.git"
    os.system(f"git clone {REPO_URL}")
    sys.path.insert(0, os.path.abspath("world-model-jepa/src"))

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available():
    print("NOTE: Runtime > Change runtime type > T4 GPU for the training notebooks.")


In [ ]:
import torch
from torch.utils.data import DataLoader
from vjepa_mini import VJEPAConfig, MovingMNIST, AttentiveProbe, train_probe
from vjepa_mini.data.moving_mnist import collate_clips
from vjepa_mini.models.vit import VideoViT
from vjepa_mini.utils import set_seed, load_checkpoint

set_seed(0)
ckpt = load_checkpoint("./checkpoints/vjepa_final.pt")
cfg: VJEPAConfig = ckpt["cfg"]

def make_encoder(state=None):
    enc = VideoViT(img_size=cfg.img_size, num_frames=cfg.num_frames,
                   in_channels=cfg.in_channels, tubelet_t=cfg.tubelet_t,
                   patch_size=cfg.patch_size, dim=cfg.enc_dim,
                   depth=cfg.enc_depth, num_heads=cfg.enc_heads).to(device)
    if state is not None: enc.load_state_dict(state)
    for p in enc.parameters(): p.requires_grad = False
    return enc.eval()

encoders = {
    "vjepa (EMA)":    make_encoder(ckpt["target_encoder"]),
    "vjepa (online)": make_encoder(ckpt["encoder"]),
    "random init":    make_encoder(None),
}
print("loaded checkpoint from step", ckpt["step"])

## Fixed probe datasets

Seeded generators so every encoder sees the *same* clips — paired comparison, lower variance. 8000 train / 2000 val clips.

In [ ]:
train_ds = MovingMNIST(num_frames=cfg.num_frames, img_size=cfg.img_size,
                       seed=1, length=8000)
val_ds   = MovingMNIST(num_frames=cfg.num_frames, img_size=cfg.img_size, train=False,
                       seed=2, length=2000)
train_dl = DataLoader(train_ds, batch_size=128, collate_fn=collate_clips, num_workers=2)
val_dl   = DataLoader(val_ds, batch_size=128, collate_fn=collate_clips, num_workers=2)

digit_label     = lambda meta: meta["digit"][:, 0]       # first (only) digit
direction_label = lambda meta: meta["direction"][:, 0]

## The main table

Feature extraction runs once per encoder (~2 min each), probes train on cached features (fast). **Predict your table before running it** — seriously, write the six numbers down.

In [ ]:
results = {}
for name, enc in encoders.items():
    results[name] = {}
    for task, label_fn, n_cls in [("digit", digit_label, 10),
                                  ("direction", direction_label, 8)]:
        print(f"=== {name} / {task} ===")
        probe, hist = train_probe(enc, train_dl, val_dl, label_fn, n_cls,
                                  device, epochs=10, verbose=False)
        results[name][task] = max(hist["val_acc"])
        print(f"    best val acc: {results[name][task]:.3f}")

print(f"\n{'encoder':<16}{'digit (chance .10)':<22}{'direction (chance .125)'}")
for name, r in results.items():
    print(f"{name:<16}{r['digit']:<22.3f}{r['direction']:.3f}")

## Reading the table

The paper-shaped conclusions to check against your numbers:

1. **vjepa >> random on BOTH tasks** → feature prediction alone produced a versatile representation. This is the core claim of the paper reproduced at 1/100 scale on a free GPU. If direction barely beats random, your model learned appearance but not motion — usually a masking or training-length issue.
2. **EMA vs online encoder**: the paper evaluates the EMA one. Is yours better? By a seed-noise margin or clearly?
3. **Random features are not at chance** — they never are (random projections preserve a lot). Improvement *over* random is the honest signal.

## Mean pooling vs attentive pooling (the App. E effect)

Prediction: which task suffers more from mean pooling, and why? (docs/06 §6.3)

In [ ]:
import torch.nn as nn, torch.nn.functional as F
from vjepa_mini.eval.attentive_probe import extract_features

def meanpool_probe(enc, label_fn, n_cls, epochs=30):
    xtr, ytr = extract_features(enc, train_dl, device, label_fn)
    xva, yva = extract_features(enc, val_dl, device, label_fn)
    xtr, xva = xtr.mean(1), xva.mean(1)                    # [B, D] pooled
    clf = nn.Linear(xtr.shape[1], n_cls).to(device)
    o = torch.optim.AdamW(clf.parameters(), lr=1e-2)
    xtr, ytr = xtr.to(device), ytr.to(device)
    for _ in range(epochs):
        o.zero_grad(); F.cross_entropy(clf(xtr), ytr).backward(); o.step()
    with torch.no_grad():
        return (clf(xva.to(device)).argmax(1).cpu() == yva).float().mean().item()

enc = encoders["vjepa (EMA)"]
for task, fn, n in [("digit", digit_label, 10), ("direction", direction_label, 8)]:
    mp = meanpool_probe(enc, fn, n)
    print(f"{task:<10} mean-pool {mp:.3f}   vs attentive {results['vjepa (EMA)'][task]:.3f}")
# Expect the direction gap >> digit gap: motion lives in WHICH tokens changed
# WHERE; averaging destroys exactly that structure.

## Qualitative: watch the features track the digit

Two views: PCA feature maps over time, and a "feature tracker" — cosine similarity between one digit token at t'=0 and every token later.

In [ ]:
import matplotlib.pyplot as plt
from vjepa_mini.eval.visualize import pca_feature_map, plot_pca_map, show_clip

grid = (cfg.grid_t, cfg.grid_h, cfg.grid_w)
clip, meta = MovingMNIST(seed=7)[0]
show_clip(clip, title=f"digit {meta['digit'].item()}")
plot_pca_map(pca_feature_map(enc, clip, device, grid), "V-JEPA features (PCA)")

@torch.no_grad()
def feature_tracker(enc, clip):
    toks = enc(clip.unsqueeze(0).to(device))[0].cpu()      # [N, D]
    toks = F.normalize(toks, dim=1).reshape(*grid, -1)     # [T',H',W',D]
    # reference: the brightest token (digit) in slice 0
    energy = clip[0, :cfg.tubelet_t].mean(0)
    pooled = F.avg_pool2d(energy[None], cfg.patch_size)[0]
    ref_h, ref_w = divmod(pooled.argmax().item(), grid[2])
    ref = toks[0, ref_h, ref_w]
    sim = (toks @ ref)                                     # [T',H',W']
    fig, axes = plt.subplots(1, grid[0], figsize=(2.2*grid[0], 2.4))
    for t in range(grid[0]):
        axes[t].imshow(sim[t], vmin=-1, vmax=1, cmap="RdBu_r")
        axes[t].set_axis_off(); axes[t].set_title(f"t'={t}", fontsize=8)
    fig.suptitle("cosine sim to the digit token at t'=0 — a free object tracker?")
    plt.tight_layout(); plt.show()

feature_tracker(enc, clip)

## Your first mini research report

You now have everything a real study has: a hypothesis (feature prediction → versatile features), an implementation, controlled baselines, quantitative + qualitative results. Write it up — half a page:

1. table of six numbers (+ seed variance if you ran the exercise),
2. one paragraph: do the results support the paper's claim at this scale?
3. one anomaly you noticed and a hypothesis for it,
4. the next experiment you would run.

That artifact is step one of `docs/07_research_gaps_and_your_path.md` — which is where you go next. Pick a gap (P1–P5), and you have a research project with this repo as the harness.

## Exercises

1. **Seed variance**: rerun the main table with probe seeds {0,1,2}. Report mean±std. Which conclusions survive the error bars?
2. **Layer-wise probing**: probe each encoder block's output. Where does direction info peak vs digit info? (Register a forward hook, or return intermediates.)
3. **Low-shot** (paper §7.3 in miniature): probes on 10% / 5% / 1% of the 8000 training clips. Plot accuracy vs labels for vjepa and random — whose curve degrades more gracefully?
4. **Speed regression**: probe for `speed` with MSE (swap the head + loss). Motion magnitude, not just direction — is it there?
5. **Challenge — Figure 7 in miniature**: train a small deconv decoder from frozen features to pixels; render *predicted* target features for a masked clip. What renders sharply, what blurs, and what does that tell you about the information the encoder chose to keep?